In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Early-Sepsis-Detection"

Mounted at /content/drive
/content/drive/MyDrive/Early-Sepsis-Detection


In [2]:
!ls /content/drive/MyDrive

 1ST.webm.gvid			      Early-Sepsis-Detection
'AN SURGICAL 2 RECORD ALL 1.gsheet'  'IBM Data Analysis'
 Classroom			     'ML Model Project'
'Colab Notebooks'		      sepsis_data


In [ ]:
!find /content/drive/MyDrive -iname "p000001.psv" 2>/dev/null

/content/drive/MyDrive/sepsis_data/raw/training_setA/p000001.psv


In [3]:
!mkdir -p /content/local_data/raw/training_setA /content/local_data/raw/training_setB
!rsync -a --info=progress2 /content/drive/MyDrive/sepsis_data/raw/training_setA/ /content/local_data/raw/training_setA/
!rsync -a --info=progress2 /content/drive/MyDrive/sepsis_data/raw/training_setB/ /content/local_data/raw/training_setB/

    131,388,704 100%  245.48kB/s    0:08:42 (xfr#20336, to-chk=0/20337)
      1,313,161   1%    4.59kB/s    0:04:38 (xfr#210, to-chk=19790/20001)
rsync error: received SIGINT, SIGTERM, or SIGHUP (code 20) at rsync.c(716) [sender=3.2.7]
rsync: [generator] write error: Broken pipe (32)
rsync error: received SIGINT, SIGTERM, or SIGHUP (code 20) at io.c(519) [receiver=3.2.7]
rsync: connection unexpectedly closed (285465 bytes received so far) [generator]


In [4]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Early-Sepsis-Detection"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Early-Sepsis-Detection


# 02 — Data Cleaning
### Early Sepsis Detection — Phase 3

This notebook applies the documented cleaning decisions in `src/data_cleaning.py`.
Every decision below is based on the **actual full-dataset results** obtained from
`01_data_understanding.ipynb` (40,336 patients, 1,552,210 hourly observations,
0 duplicate rows, 7.269% patient-level sepsis prevalence).

Raw data in `data/raw/` is never modified — this notebook reads it, cleans it in
memory, and writes the result to `data/interim/`.


In [5]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src import data_loader as dl
from src import data_cleaning as dc

pd.set_option("display.max_columns", 60)


## 1. Load the raw combined dataframe

If you already ran `01_data_understanding.ipynb` with `SAMPLE_N_PATIENTS = None`
and `save_parquet=True`, the full dataset was cached to
`data/interim/sepsis_hourly_long.parquet` — loading from there is much faster
than re-reading 40,336 `.psv` files from Drive.


In [6]:
if config.INTERIM_LONG_PARQUET.exists():
    print("Loading cached parquet (fast path)...")
    df = pd.read_parquet(config.INTERIM_LONG_PARQUET)
else:
    print("No cache found — loading raw .psv files (this will take longer)...")
    df = dl.load_combined_dataframe(sample_n_patients=None, save_parquet=True)

print("Loaded shape:", df.shape)
print("Unique patients:", df[config.PATIENT_ID_COL].nunique())


Loading cached parquet (fast path)...
Loaded shape: (1552210, 43)
Unique patients: 40336


## 2. Decision 1 — Implausible physiological values

**What?** Check vitals/age against known physiologically plausible ranges
(e.g. HR 0-300 bpm, Temp 25-45°C, O2Sat 0-100%).

**Why?** Values outside these ranges are almost certainly sensor artifacts or
data-entry errors, not real physiology.

**How?** `flag_implausible_values()` counts violations per column without
modifying anything; `clean_implausible_values()` then sets only the genuinely
impossible values to NaN (so they flow into the same imputation pipeline as
true missing data, rather than being silently clipped).


In [7]:
implausible_report = dc.flag_implausible_values(df)
display(implausible_report)


,column,n_implausible,range_checked
0,HR,0,"[0, 300]"
1,O2Sat,0,"[0, 100]"
2,Temp,6,"[25, 45]"
3,SBP,0,"[0, 300]"
4,MAP,201,"[0, 250]"
5,DBP,103,"[0, 200]"
6,Resp,0,"[0, 100]"
7,Age,0,"[0, 120]"


In [8]:
df = dc.clean_implausible_values(df)
print("Implausible values converted to NaN where found (see counts above).")


2026-09-13 20:32:04,494 | WARNING | src.data_cleaning | Setting 6 implausible 'Temp' values (outside [25, 45]) to NaN.
2026-09-13 20:32:04,539 | WARNING | src.data_cleaning | Setting 201 implausible 'MAP' values (outside [0, 250]) to NaN.
2026-09-13 20:32:04,558 | WARNING | src.data_cleaning | Setting 103 implausible 'DBP' values (outside [0, 200]) to NaN.


Implausible values converted to NaN where found (see counts above).


## 3. Decision 2 — Duplicate records

**Result (Phase 2, full dataset):** 0 fully duplicated rows out of 1,552,210.
Re-verified here on the loaded dataframe as a safety check.


In [9]:
n_dupes = dc.check_duplicates(df)
print(f"Duplicate rows: {n_dupes}")


2026-09-13 20:32:09,149 | INFO | src.data_cleaning | No duplicate rows found (matches Phase 2 finding of 0/1,552,210).
INFO:src.data_cleaning:No duplicate rows found (matches Phase 2 finding of 0/1,552,210).


Duplicate rows: 0


## 4. Decision 3 — Constant features

**What?** Confirm no column is constant (would carry zero information).

**Result:** Based on Phase 2's unique-value counts, no column had `n_unique <= 1`.
Re-checked here on the actual loaded data.


In [10]:
constant_cols = dc.find_constant_features(df)
print("Constant columns found:", constant_cols if constant_cols else "None")


Constant columns found: None


## 5. Decision 4 — Unit1 / Unit2 missingness (39.43% each, identical rows)

**What?** Unit1 and Unit2 are missing for the exact same 611,960 rows.

**Why?** This looks like a per-patient / per-source-set pattern (ICU-unit info
not recorded for a subset of patients), not random missingness. We check this
against `source_set` before deciding on treatment.

**Treatment:** Explicit "Unknown" encoding (sentinel = -1) + a `Unit{1,2}_known`
indicator flag — never an imputed/fabricated unit.


In [11]:
unit_diagnosis = dc.diagnose_unit_missingness(df)
display(unit_diagnosis)


,n_missing_rows,n_rows,pct_missing
source_set,,,
A,386165,790215,0.488683
B,225795,761995,0.296321


In [12]:
df = dc.clean_unit_columns(df)
print("Unit1/Unit2 missingness encoded as explicit 'Unknown' (-1) + *_known indicator flags.")
df[["Unit1", "Unit2", "Unit1_known", "Unit2_known"]].head()


Unit1/Unit2 missingness encoded as explicit 'Unknown' (-1) + *_known indicator flags.


,Unit1,Unit2,Unit1_known,Unit2_known
0,-1.0,-1.0,0,0
1,-1.0,-1.0,0,0
2,-1.0,-1.0,0,0
3,-1.0,-1.0,0,0
4,-1.0,-1.0,0,0


## 6. Decision 5 — HospAdmTime (8 rows missing, ~0.0005%)

**What?** A negligible number of rows are missing HospAdmTime, which is
constant per patient.

**Treatment:** Recover via within-patient forward/backward fill first (covers
any patient who has the value recorded on at least one other row); true
"never recorded for this patient" cases are left for training-set-only
imputation in Phase 11.


In [13]:
before_missing = df["HospAdmTime"].isna().sum()
df = dc.clean_hospadmtime(df)
after_missing = df["HospAdmTime"].isna().sum()
print(f"HospAdmTime missing before: {before_missing}, after within-patient fill: {after_missing}")


HospAdmTime missing before: 8, after within-patient fill: 8


In [15]:
path = "/content/drive/MyDrive/Early-Sepsis-Detection/src/data_cleaning.py"

with open(path, "r") as f:
    content = f.read()

# Fix 1: Remove duplicate "Temp" from MODERATE_MISSINGNESS_VARS
content = content.replace(
    'MODERATE_MISSINGNESS_VARS = ["Glucose", "Temp"]',
    'MODERATE_MISSINGNESS_VARS = ["Glucose"]'
)

# Fix 2: De-duplicate columns list defensively inside forward_fill_within_patient
content = content.replace(
    '    out = df.copy()\n    out[columns] = out.groupby(config.PATIENT_ID_COL, sort=False)[columns].ffill()\n    return out',
    '    columns = list(dict.fromkeys(columns))\n    out = df.copy()\n    out[columns] = out.groupby(config.PATIENT_ID_COL, sort=False)[columns].ffill()\n    return out'
)

with open(path, "w") as f:
    f.write(content)

print("Patched successfully.")

Patched successfully.


In [16]:
import importlib
from src import data_cleaning as dc
importlib.reload(dc)

<module 'src.data_cleaning' from '/content/drive/MyDrive/Early-Sepsis-Detection/src/data_cleaning.py'>

## 7. Decision 6 — Forward-fill (LOCF) for vitals and laboratory values

**What?** Apply last-observation-carried-forward, strictly within each
patient, in existing (validated) ICULOS row order.

**Why this is leakage-safe:** each row only uses earlier rows of the SAME
patient — never other patients, never future timepoints.

**Why LOCF (not mean/zero-fill) here:** a vital/lab value measured at hour t
remains the patient's best-known state until the next reading; this matches
standard ICU time-series practice and the original PhysioNet 2019 baseline
approach. Extreme-sparsity labs (>90% missing) still benefit from this because
even a lab drawn once early in the stay is clinically relevant for many
subsequent hours.


In [17]:
locf_cols = dc.VITAL_SIGNS_FOR_LOCF + dc.MODERATE_MISSINGNESS_VARS + dc.EXTREME_SPARSITY_LABS
locf_cols = [c for c in locf_cols if c in df.columns]

missing_before = df[locf_cols].isna().mean().sort_values(ascending=False) * 100
df = dc.forward_fill_within_patient(df, locf_cols)
missing_after = df[locf_cols].isna().mean().sort_values(ascending=False) * 100

comparison = pd.DataFrame({"missing_pct_before_locf": missing_before.round(2),
                            "missing_pct_after_locf": missing_after.round(2)})
display(comparison)


,missing_pct_before_locf,missing_pct_after_locf
AST,98.38,69.40
Alkalinephos,98.39,69.69
BUN,93.13,20.55
BaseExcess,94.58,68.10
Bilirubin_direct,99.81,95.34
Bilirubin_total,98.51,69.79
Calcium,94.12,28.46
Chloride,95.46,53.57
Creatinine,93.90,21.40
DBP,31.35,20.65


**Note on remaining missingness after LOCF:** Any NaN remaining here means
that patient had NO measurement of that variable up to and including that
hour (e.g. the very first ICU hour before any lab was drawn, or a lab never
ordered for that patient at all). These remaining gaps are intentionally
**not** filled in this notebook — they are handled by the training-set-only
imputer in Phase 11 (`src/preprocessing.py`), to avoid any risk of leaking
validation/test information into an imputation statistic computed here.


## 8. Save cleaned interim dataset

In [18]:
interim_path = config.INTERIM_DIR / "sepsis_hourly_cleaned.parquet"
df.to_parquet(interim_path, index=False)
print(f"Saved cleaned dataframe to {interim_path}")
print("Shape:", df.shape)


Saved cleaned dataframe to /content/drive/MyDrive/Early-Sepsis-Detection/data/interim/sepsis_hourly_cleaned.parquet
Shape: (1552210, 45)


---
### What to send back to Claude after running this notebook

- The Section 2 implausible-values table (any nonzero counts especially)
- The Section 5 Unit1/Unit2-by-source_set table
- The Section 7 before/after LOCF missingness comparison table
- Confirmation the parquet saved successfully in Section 8

With that, we move to **Phase 4 (Exploratory Data Analysis)** — target
prevalence visuals, clinical variable distributions, and sepsis vs.
non-sepsis time-series comparisons, all using this cleaned dataset.
